In [1]:
!pip install -q bitsandbytes datasets accelerate loralib

In [2]:
import torch
import torch.nn as nn

In [3]:
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [3]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

In [5]:
import os
import bitsandbytes as bnb

In [6]:
model_name = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [7]:
i = 5
for p in model.named_parameters():
  print(p)
  i -= 1
  if i < 0:
    break

('model.embed_tokens.weight', Parameter containing:
tensor([[ 0.0352, -0.0229,  0.0815,  ...,  0.0211,  0.0527, -0.0352],
        [-0.0200,  0.0522, -0.0303,  ...,  0.0028, -0.0240, -0.0173],
        [-0.0002, -0.0059,  0.0222,  ...,  0.0152, -0.0074, -0.0119],
        ...,
        [ 0.0371, -0.0237,  0.0486,  ...,  0.0075,  0.0064,  0.0134],
        [ 0.0117, -0.0410,  0.0253,  ..., -0.0024,  0.0454,  0.0219],
        [ 0.0361, -0.0262,  0.0786,  ...,  0.0215,  0.0525, -0.0366]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True))
('model.layers.0.self_attn.q_proj.weight', Parameter containing:
Parameter(Int8Params([[-12,  25, -32,  ...,  24, -46,  14],
            [ -4,  29, -47,  ...,  38, -19,   9],
            [  4,   8,  15,  ...,  -2,  32,  -3],
            ...,
            [  6,  59,  19,  ...,  37, -24,  -3],
            [ 12,  -7,  16,  ...,  21,   1,  18],
            [-20, -22,  42,  ..., -30,  66, -36]], device='cuda:0',
           dtype=torch.int8)))
('mode

In [8]:
for param in model.parameters():
  param.requires_grad = False
  if param.ndim == 1:
    param.data = param.data.to(torch.float32)

In [9]:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [10]:
class CastOutputToFloat(nn.Sequential):
  def forward(self, x):
    return super().forward(x).to(torch.float32)

model.lm_head = CastOutputToFloat(model.lm_head)

In [11]:
def print_trainable_parameters(model):
    """
  printing the number of trainable paramters in the model
  """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")

# FINETUNING

In [12]:
pip install peft

In [13]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [39]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [40]:
print_trainable_parameters(model)

trainable params: 1384448 || all params: 2615726336 || trainable%: 0.0529278610283488


In [41]:
lora_model = get_peft_model(model, config)
print_trainable_parameters(lora_model)

trainable params: 2768896 || all params: 2617110784 || trainable%: 0.10579972452553235


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [17]:
import transformers
from datasets import load_dataset

In [18]:
data = load_dataset("Abirate/english_quotes")

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [19]:
data

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [20]:
data['train'][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator']}

In [21]:
def merge_columns(example):
  example['prediction'] = example['quote'] + " ->: " + str(example['tags'])
  return example

In [22]:
data['train'] = data['train'].map(merge_columns)
data['train']['prediction'][0]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

"“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']"

In [23]:
data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [24]:
data

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags', 'prediction', 'input_ids', 'attention_mask'],
        num_rows: 2508
    })
})

In [25]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): Linear8bitLt(in_features=2304, out_features=2048, bias=False)
              (k_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2304, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): Para

In [42]:
trainer = transformers.Trainer(
    model = lora_model,
    train_dataset = data['train'],
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 10,
        max_steps = 50,
        learning_rate = 2e-4,
        fp16 = False,
        logging_steps = 1,
        output_dir = 'outputs'
    ),
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm = False)
)
model.config.use_cache = False
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
1,3.757522
2,3.587021
3,4.161676
4,5.038037
5,3.504489
6,2.688970
7,2.462592
8,3.806912
9,3.208262
10,2.918525


TrainOutput(global_step=50, training_loss=2.5727405619621275, metrics={'train_runtime': 49.5972, 'train_samples_per_second': 1.008, 'train_steps_per_second': 1.008, 'total_flos': 35809993752576.0, 'train_loss': 2.5727405619621275, 'epoch': 0.019936204146730464})

In [43]:
lora_model.push_to_hub("afreediz/gemma-2-2b-it-tagger-test-adapter",
                      commit_message = "Testing Lora Training method",
                      private=False)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   5%|5         |  565kB / 11.1MB            

CommitInfo(commit_url='https://huggingface.co/afreediz/gemma-2-2b-it-tagger-test-adapter/commit/341f126c55afaa754f28df3a15860725494f3686', commit_message='Testing Lora Training method', commit_description='', oid='341f126c55afaa754f28df3a15860725494f3686', pr_url=None, repo_url=RepoUrl('https://huggingface.co/afreediz/gemma-2-2b-it-tagger-test-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='afreediz/gemma-2-2b-it-tagger-test-adapter'), pr_revision=None, pr_num=None)

In [59]:
# 3. For inference, use the chat template for instruct models
message = "Be yourself; everyone else is already taken."
text_ids = tokenizer(message, return_tensors="pt").to(model.device)

In [60]:
with torch.no_grad():
    res = model.generate(
        **text_ids,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
    )

In [61]:
print(tokenizer.decode(res[0], skip_special_tokens=True))

Be yourself; everyone else is already taken. -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> -> <- <- <- <- <- <- <- <- <- <- <- <- <- <- <-


In [62]:
with torch.no_grad():
    lora_res = lora_model.generate(
        **text_ids,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        top_p=0.9,
    )

In [64]:
prompt_len = len(tokenizer.encode(message))

In [65]:
print(tokenizer.decode(lora_res[0][prompt_len:], skip_special_tokens=True))

 -->
*
























































































































































































































































                                                                                                                                         

 
```````````'`'ssssssssssssssssssssssssssssssssssssswwwwwwwwwwwwwwwwwwwwwwww


In [54]:
print(tokenizer.decode(lora_res[0], skip_special_tokens=True))

user
be yourself
model
PleaseThankYouYouYou You




















````````` ``` ``` ```  ** ** ** **  . . . . ... ... ...


















































































































































































































